# Brazilian E-Commerce Data Analysis

## Project Overview

This project analyzes a real-world Brazilian e-commerce dataset (Olist) containing multi-table transactional data.

The goal of this phase is:

- To understand dataset structure
- To audit and classify null values
- To validate business logic consistency
- To prepare clean analytical tables for further analysis

This dataset contains multiple relational tables such as:
- Orders
- Order Items
- Customers
- Products
- Payments
- Reviews
- Sellers
- Geolocation
- Category Translation

This notebook focuses on foundational cleaning and data validation.

In [16]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading

The Olist dataset is relational in structure.
Each CSV represents a different business entity.

We load all tables separately to preserve relational integrity.

In [17]:
base_path = r"/home/srikanth/Documents/Brazilian E-Commerce Public Dataset by Olist/"

orders = pd.read_csv(base_path + "olist_orders_dataset.csv")
order_items = pd.read_csv(base_path + "olist_order_items_dataset.csv")
customers = pd.read_csv(base_path + "olist_customers_dataset.csv")
products = pd.read_csv(base_path + "olist_products_dataset.csv")
payments = pd.read_csv(base_path + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(base_path + "olist_order_reviews_dataset.csv")
sellers = pd.read_csv(base_path + "olist_sellers_dataset.csv")
geolocation = pd.read_csv(base_path + "olist_geolocation_dataset.csv")
translation = pd.read_csv(base_path + "product_category_name_translation.csv")

### Insight

The dataset consists of 9 interconnected tables.

This confirms we are working with a relational dataset,
not a flat single-table dataset.

Proper grain handling will be critical.

## Null Value Analysis

Before performing any grouping or KPI calculation,
we audit null values across all tables.

We classify nulls into:

1. Business lifecycle nulls
2. Optional informational nulls
3. True data quality issues

In [18]:
def null_summary(df,name):
 print(f"\n{name}")
 nulls=df.isnull().sum()
 nulls=nulls[nulls > 0]
 print(nulls if len(nulls) > 0 else "no null values")

null_summary(orders, "Orders")
null_summary(order_items, "Order Items")
null_summary(customers, "Customers")
null_summary(products, "Products")
null_summary(payments, "Payments")
null_summary(reviews, "Reviews")


Orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

Order Items
no null values

Customers
no null values

Products
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Payments
no null values

Reviews
review_comment_title      87656
review_comment_message    58247
dtype: int64


In [19]:
orders[['order_status',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date']].groupby('order_status').count()

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,2,0,0
canceled,484,75,6
created,0,0,0
delivered,96464,96476,96470
invoiced,314,0,0
processing,301,0,0
shipped,1107,1107,0
unavailable,609,0,0


### Insight

Orders table contains null values in:

- order_approved_at
- order_delivered_carrier_date
- order_delivered_customer_date

These nulls are NOT data errors.
They represent different stages of the order lifecycle.

For example:
- Canceled orders
- Processing orders
- Shipped but not delivered orders

These values should NOT be filled or dropped.
They will be handled using order_status filtering.

## Product Metadata Cleaning

The products table contains missing metadata values
in product category and dimension fields.

In [20]:
products['product_category_name'] = (
    products['product_category_name']
    .fillna('unknown')
)

### Insight

610 products were missing category metadata.

Instead of dropping rows,
we labeled them as "unknown" to preserve transactional integrity.

Other dimension nulls were minimal (2 rows) and
do not impact current analytical goals.

In [21]:
orders['order_id'].isnull().sum()
order_items['order_id'].isnull().sum()
customers['customer_id'].isnull().sum()

0

### Insight

There are noo null values in unique values 

# Revenue Engineering & Order Monetization

## Objective

The objective of this phase is to construct a financially accurate
revenue model from a multi-table relational dataset.

Since revenue exists at the item level,
we must correctly aggregate and align data grain
before performing any financial analysis.

## Revenue Source Identification

The Orders table does not contain revenue information.

Revenue is recorded at the item level
inside the Order Items table.

Each row in Order Items represents:
- One product inside an order
- With price and freight_value

To compute total revenue per order,
we must first construct item-level revenue.

In [22]:
order_items['total_price'] = ( 
    order_items['price']+order_items['freight_value']
)

### Insight

We created total_price to represent
the true revenue contribution of each item.

This ensures shipping costs are included in revenue.

## Grain Alignment Strategy

Orders Table:
1 row = 1 order

Order Items Table:
1 row = 1 product inside an order

Direct merging would cause revenue duplication.

To resolve this,
we aggregate revenue at the order level
before merging.

In [23]:
order_revenue = (
    order_items
    .groupby('order_id')['total_price']
    .sum()
    .reset_index()
)

order_revenue.rename(
    columns={'total_price': 'order_total_revenue'},
    inplace=True
)

### Insight

This aggregation step ensures:

- One revenue value per order
- No duplication during joins
- Financial accuracy

This is critical when working with relational datasets.

## Integrating Revenue with Orders

We merge order-level revenue into the Orders table
to create a revenue-enabled analytical dataset.

In [24]:
orders_with_revenue = orders.merge(
    order_revenue,
    on='order_id',
    how='left'
)

### Insight

A left join preserves all orders
while attaching revenue where available.

The dataset is now revenue-ready.

## Realized Revenue Filtering

Revenue should only reflect completed transactions.

Orders marked as:
- canceled
- unavailable

do not represent realized business revenue.

We therefore filter only delivered orders.

In [25]:
delivered_orders = orders_with_revenue[
    orders_with_revenue['order_status'] == 'delivered'
]

### Insight

By restricting analysis to delivered orders,
we ensure:

- Revenue reflects completed transactions
- Financial KPIs remain accurate
- Cancellations do not inflate performance metrics

## Summary

At the end of Phase :

- Revenue has been engineered correctly
- Data grain mismatch has been resolved
- A clean delivered-orders dataset has been created
- Financial analysis can now be performed safely

The dataset is ready for time-based and performance analysis.

## Time Feature Engineering

To perform time-based analysis,
we extract temporal attributes
from the order_purchase_timestamp column.

This enables grouping by year and month.

In [26]:
orders_with_revenue['order_purchase_timestamp'] = pd.to_datetime(
    orders_with_revenue['order_purchase_timestamp'],
    errors='coerce'
)

orders_with_revenue['order_year'] = (
    orders_with_revenue['order_purchase_timestamp'].dt.year
)

orders_with_revenue['order_month'] = (
    orders_with_revenue['order_purchase_timestamp'].dt.month
)

### Insight

Extracting time attributes allows:

- Annual performance comparison
- Monthly seasonality detection
- Structured time-based aggregation

Time features are essential for business growth analysis.

## Filtering Delivered Orders

Revenue should reflect completed transactions only.

Canceled and unavailable orders
do not represent realized revenue.

We therefore restrict analysis
to delivered orders.

In [36]:
delivered_orders = orders_with_revenue[
    orders_with_revenue['order_status'] == 'delivered'
].copy()

### Insight

Filtering delivered orders ensures:

- Financial accuracy
- Realized revenue measurement
- Clean performance metrics

## Yearly Revenue Analysis

We calculate total revenue per year
to observe overall business growth.

In [28]:
yearly_revenue = (
    delivered_orders
    .groupby('order_year')['order_total_revenue']
    .sum()
)
print(yearly_revenue)

order_year
2016      46653.74
2017    6921535.24
2018    8451584.77
Name: order_total_revenue, dtype: float64


### Insight

Yearly revenue results indicate:

- The year with highest revenue
- The speed of business expansion
- Evidence of scaling trend

Revenue growth reflects market penetration
and operational maturity.

## Yearly Order Volume

Revenue growth may result from:

- Increased number of orders
- Higher average order value
- Or both

We calculate yearly unique order counts.

In [29]:
yearly_orders = (
    delivered_orders
    .groupby('order_year')['order_id']
    .nunique()
)

print(yearly_orders)

order_year
2016      267
2017    43428
2018    52783
Name: order_id, dtype: int64


### Insight

Order volume trend helps determine:

- Customer acquisition growth
- Expansion speed
- Market adoption rate

Comparing revenue and order growth
reveals whether performance improvement
comes from scale or pricing.

## Monthly Revenue Seasonality

We evaluate revenue distribution by month
to detect seasonal demand fluctuations.

In [30]:
monthly_revenue = (
    delivered_orders
    .groupby('order_month')['order_total_revenue']
    .sum()
)

print(monthly_revenue.sort_index())

order_month
1     1205369.83
2     1237407.73
3     1534929.19
4     1523691.33
5     1695625.92
6     1502028.66
7     1594106.36
8     1631324.00
9      701220.95
10     797607.67
11    1153364.20
12     843097.91
Name: order_total_revenue, dtype: float64


### Insight

Monthly revenue analysis identifies:

- Peak demand months
- Slow business periods
- Seasonal consumer behavior

This supports marketing and inventory planning decisions.

## Summary

At the end of this phase, we have:

- Measured annual revenue growth
- Evaluated order volume expansion
- Identified seasonal demand patterns
- Prepared the foundation for growth storytelling

The business trajectory can now be interpreted
from both revenue and operational perspectives.

# Operational & Business Intelligence

## Delivery Performance Analysis

Efficient delivery is critical in e-commerce.

In this section, we analyze:

- Average delivery time
- Delivery time distribution
- Late delivery percentage
- Operational reliability

Delivery performance directly impacts
customer satisfaction and repeat purchases.

In [39]:
delivered_orders['order_delivered_customer_date'] = pd.to_datetime(
    delivered_orders['order_delivered_customer_date'],
    errors='coerce'
)

In [40]:
    print(delivered_orders[['order_purchase_timestamp',
                        'order_delivered_customer_date']].dtypes)

order_purchase_timestamp         datetime64[ns]
order_delivered_customer_date    datetime64[ns]
dtype: object


In [41]:
delivered_orders['delivery_days'] = (
    delivered_orders['order_delivered_customer_date'] -
    delivered_orders['order_purchase_timestamp']
).dt.days

### Insight

Delivery days measure the operational efficiency
from order placement to customer delivery.

This metric evaluates logistics performance.

In [42]:
avg_delivery_time = delivered_orders['delivery_days'].mean()
print("Average Delivery Time (days):", round(avg_delivery_time, 2))

Average Delivery Time (days): 12.09


### Insight

The average delivery time indicates
overall logistics efficiency.

Lower delivery time improves customer experience
and increases satisfaction.

In [43]:
delivered_orders['late_delivery'] = (
    delivered_orders['order_delivered_customer_date'] >
    delivered_orders['order_estimated_delivery_date']
)

late_percentage = delivered_orders['late_delivery'].mean() * 100
print("Late Delivery Percentage:", round(late_percentage, 2), "%")

Late Delivery Percentage: 8.11 %


### Insight

Late delivery percentage reflects
logistics reliability.

A high late percentage may negatively impact
customer trust and review ratings.

In [44]:
delivery_stats = delivered_orders['delivery_days'].describe()
print(delivery_stats)

count    96470.000000
mean        12.093604
std          9.551380
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64


### Insight

Delivery distribution shows:

- Typical delivery duration
- Variability in logistics performance
- Presence of extreme outliers

This helps identify operational inconsistencies.

In [45]:
delivery_reviews = delivered_orders.merge(
    reviews[['order_id', 'review_score']],
    on='order_id',
    how='left'
)

late_review_comparison = (
    delivery_reviews
    .groupby('late_delivery')['review_score']
    .mean()
)

print(late_review_comparison)

late_delivery
False    4.293737
True     2.566494
Name: review_score, dtype: float64


### Insight

Comparing review scores between
late and on-time deliveries reveals:

- Whether logistics delays impact customer satisfaction
- Operational influence on brand perception

In [46]:
delivery_by_year = (
    delivered_orders
    .groupby('order_year')['delivery_days']
    .mean()
)

print(delivery_by_year)

order_year
2016    19.209738
2017    12.541335
2018    11.689202
Name: delivery_days, dtype: float64


### Insight

Yearly delivery averages reveal:

- Whether logistics improved over time
- Whether scaling impacted performance
- Operational maturity trend

## Category-Level Revenue Contribution

Revenue growth alone is not enough.

We must identify:

- Which product categories drive revenue
- Revenue concentration levels
- High-performing segments
- Strategic product dominance

This helps understand product portfolio strength.

In [47]:
order_items_products = order_items.merge(
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)

### Insight

We merge product category information
to associate each sold item with its category.

This enables revenue analysis at the product segment level.

In [48]:
delivered_items = order_items_products.merge(
    delivered_orders[['order_id']],
    on='order_id',
    how='inner'
)

In [49]:
category_revenue = (
    delivered_items
    .groupby('product_category_name')['total_price']
    .sum()
    .sort_values(ascending=False)
)

print(category_revenue.head(10))

product_category_name
beleza_saude              1412089.53
relogios_presentes        1264333.12
cama_mesa_banho           1225209.26
esporte_lazer             1118256.91
informatica_acessorios    1032723.77
moveis_decoracao           880329.92
utilidades_domesticas      758392.25
cool_stuff                 691680.89
automotivo                 669454.75
ferramentas_jardim         567145.68
Name: total_price, dtype: float64


### Insight

Top revenue-generating categories indicate:

- Core business strengths
- Customer demand concentration
- Strategic product focus areas

High revenue concentration suggests
dependency on specific product segments.

In [51]:
total_category_revenue = category_revenue.sum()

category_percentage = (
    category_revenue / total_category_revenue
) * 100

print(category_percentage.head(10))

product_category_name
beleza_saude              9.157654
relogios_presentes        8.199427
cama_mesa_banho           7.945702
esporte_lazer             7.252097
informatica_acessorios    6.697399
moveis_decoracao          5.709098
utilidades_domesticas     4.918310
cool_stuff                4.485675
automotivo                4.341534
ferramentas_jardim        3.678042
Name: total_price, dtype: float64


### 🔎 Insight

Revenue percentage helps identify:

- Dominant categories
- Long-tail distribution
- Market diversification level

In [52]:
top5_contribution = category_percentage.head(5).sum()
print("Top 5 Categories Contribution (%):", round(top5_contribution, 2))

Top 5 Categories Contribution (%): 39.25


## Customer Intelligence Analysis

Revenue performance must be analyzed
from the customer perspective.

In this section we evaluate:

- Revenue per customer
- Repeat purchase behavior
- Average Order Value (AOV)
- Revenue distribution across customers

This helps understand customer value concentration.

In [54]:
customer_orders = delivered_orders.merge(
    customers[['customer_id', 'customer_state']],
    on='customer_id',
    how='left'
)

### Insight

Merging customer information enables:

- Regional revenue analysis
- Customer segmentation
- Behavioral insights

In [55]:
customer_revenue = (
    customer_orders
    .groupby('customer_id')['order_total_revenue']
    .sum()
    .sort_values(ascending=False)
)

print(customer_revenue.head(10))

customer_id
1617b1357756262bfa56ab541c47bc16    13664.08
ec5b2ba62e574342386871631fafd3fc     7274.88
c6e2731c5b391845f6800c97401a43a9     6929.31
f48d464a0baaea338cb25f816991ab1f     6922.21
3fd6777bbce08a352fddd04e4a7cc8f6     6726.66
05455dfa7cd02f13d132aa7a6a9729c6     6081.54
df55c14d1476a9a3467f131269c2477f     4950.34
24bbf5fd2f2e1b359ee7de94defc4a15     4764.34
3d979689f636322c62418b6346b1c6d2     4681.78
1afc82cd60e303ef09b4ef9837c9505c     4513.32
Name: order_total_revenue, dtype: float64


### Insight

Top customers by revenue highlight:

- High-value customer concentration
- Potential VIP segments
- Revenue dependency risk

In [60]:
customer_orders = delivered_orders.merge(
    customers[['customer_id', 'customer_unique_id']],
    on='customer_id',
    how='left'
)

customer_order_counts = (
    customer_orders
    .groupby('customer_unique_id')['order_id']
    .nunique()
)

repeat_customers = (customer_order_counts > 1).sum()
total_customers = customer_order_counts.count()

repeat_rate = (repeat_customers / total_customers) * 100

print("Repeat Customer Percentage:", round(repeat_rate, 2), "%")

Repeat Customer Percentage: 3.0 %


### Insight

Repeat purchase rate indicates:

- Customer retention strength
- Brand loyalty
- Sustainability of revenue growth

Low repeat rate suggests acquisition-heavy growth.

In [57]:
avg_order_value = delivered_orders['order_total_revenue'].mean()
print("Average Order Value:", round(avg_order_value, 2))

Average Order Value: 159.83


### Insight

Average Order Value (AOV) reflects:

- Pricing power
- Basket size
- Upselling effectiveness

In [59]:
state_revenue = (
    customer_orders
    .groupby('customer_state')['order_total_revenue']
    .sum()
    .sort_values(ascending=False)
)

print(state_revenue.head(10))

customer_state
SP    5769703.15
RJ    2055401.57
MG    1818891.67
RS     861472.79
PR     781708.80
SC     595127.78
BA     591137.81
DF     346123.35
GO     334212.35
ES     317657.93
Name: order_total_revenue, dtype: float64


### Insight

State-level revenue highlights:

- Geographic concentration
- Strong regional markets
- Expansion opportunities

# Final Executive Summary

This analysis reveals:

- Strong revenue growth over time
- Healthy Average Order Value (~159.83)
- Revenue concentration in São Paulo (SP)
- Measurable logistics performance
- Product revenue concentrated in key categories
- Customer retention insights

The dataset has been transformed into
a business intelligence-ready analytical model.